# Embeddings maken in Snowflake

Maakt met Snowflake Cortex (`snowflake-arctic-embed-l-v2.0`, meertalig, 1024 dimensies) een embedding van elke `NOTE` in `FRUTA.PUBLIC.RECORDS`.
De embeddings komen in een eigen tabel `RECORD_EMBEDDINGS`, te koppelen via `NOTE_ID`.

In [ ]:
CREATE OR REPLACE TABLE FRUTA.PUBLIC.RECORD_EMBEDDINGS AS
SELECT
    NOTE_ID,
    SNOWFLAKE.CORTEX.EMBED_TEXT_1024('snowflake-arctic-embed-l-v2.0', NOTE) AS EMBEDDING
FROM FRUTA.PUBLIC.RECORDS
WHERE NOTE IS NOT NULL
  AND NOTE != '';

In [ ]:
-- Controle: aantal records en aantal embeddings
SELECT
    (SELECT COUNT(*) FROM FRUTA.PUBLIC.RECORDS) AS records,
    (SELECT COUNT(*) FROM FRUTA.PUBLIC.RECORD_EMBEDDINGS) AS embeddings;

In [ ]:
-- Test: zoek de rapportages die het meest lijken op een vraag
SELECT
    r.NOTE_ID,
    r.NOTE,
    VECTOR_COSINE_SIMILARITY(
        e.EMBEDDING,
        SNOWFLAKE.CORTEX.EMBED_TEXT_1024('snowflake-arctic-embed-l-v2.0', 'Mevrouw is gevallen')
    ) AS SCORE
FROM FRUTA.PUBLIC.RECORD_EMBEDDINGS e
JOIN FRUTA.PUBLIC.RECORDS r ON r.NOTE_ID = e.NOTE_ID
ORDER BY SCORE DESC
LIMIT 10;